In [1]:
import joblib
import pandas as pd
import numpy as np
from datetime import timedelta

In [2]:
sellers = pd.read_csv('../data/raw/sellers.csv')
items = pd.read_csv('../data/raw/order_items.csv')
geolocation = pd.read_csv('../data/raw/geolocation.csv')
category = pd.read_csv('../data/raw/product_category_name_translation.csv')
orders = pd.read_csv('../data/raw/orders.csv')

payment = pd.read_csv('../data/raw/order_payments.csv')
products = pd.read_csv('../data/raw/products.csv')

In [3]:
unique_geo = geolocation.groupby('geolocation_zip_code_prefix').agg(lat = ('geolocation_lat','median'),
                                                                    lng = ('geolocation_lng','median'),
                                                                   state = ('geolocation_state', lambda x: x.mode().iloc[0])).reset_index()


In [4]:
unique_geo

,geolocation_zip_code_prefix,lat,lng,state
0,1001,-23.550107,-46.634027,SP
1,1002,-23.548228,-46.635247,SP
2,1003,-23.548976,-46.635318,SP
3,1004,-23.549550,-46.634771,SP
4,1005,-23.549690,-46.636532,SP
...,...,...,...,...
19010,99960,-27.953797,-52.029641,RS
19011,99965,-28.179542,-52.035551,RS
19012,99970,-28.343257,-51.875470,RS
19013,99980,-28.388342,-51.846871,RS


In [5]:
seller_geo = sellers.merge(unique_geo.rename(columns ={'lat':'seller_latitude','lng':'seller_longitude'}), 
                                left_on='seller_zip_code_prefix',right_on='geolocation_zip_code_prefix',how='left')


## Product 

In [6]:
products['product_volume_cm3'] = products['product_width_cm'] * products['product_height_cm'] *products['product_length_cm']  

In [7]:
products_agg = products[['product_id','product_category_name','product_volume_cm3','product_weight_g']].merge(category,on='product_category_name',how='left')

In [8]:
product_items_agg =  items[['product_id','seller_id',]].merge(products_agg[['product_id','product_category_name_english','product_volume_cm3','product_weight_g']],on='product_id',how='left')

In [9]:
product_items_agg

,product_id,seller_id,product_category_name_english,product_volume_cm3,product_weight_g
0,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,cool_stuff,3528.0,650.0
1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,pet_shop,60000.0,30000.0
2,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,furniture_decor,14157.0,3050.0
3,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,perfumery,2400.0,200.0
4,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,garden_tools,42000.0,3750.0
...,...,...,...,...,...
112645,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,housewares,53400.0,10150.0
112646,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,computers_accessories,44460.0,8950.0
112647,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,sports_leisure,9576.0,967.0
112648,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,computers_accessories,8000.0,100.0


In [10]:
shopping_dataset = product_items_agg.merge(seller_geo[['seller_id','seller_state','seller_latitude','seller_longitude']],
                       on='seller_id',how='left')

In [11]:
shopping_dataset.shape

(112650, 8)

In [12]:
shopping_dataset_unique = shopping_dataset.drop_duplicates(subset=['seller_id','product_id']).rename(columns={'product_category_name_english':'category'})

In [13]:
shopping_cols=['category','product_id','seller_id','seller_state','seller_latitude','seller_longitude','product_volume_cm3','product_weight_g'] 
shopping_dataset_unique = shopping_dataset_unique[shopping_cols]

In [14]:
shopping_dataset_unique['category_formated'] = shopping_dataset_unique['category'].str.replace('_',' ').str.capitalize()
shopping_dataset_unique.rename(columns={'category':'raw_category'},inplace=True)

In [15]:
shopping_dataset_unique.dropna(inplace=True)

In [16]:
order_cols = ['order_id','order_purchase_timestamp','order_delivered_carrier_date','order_delivered_customer_date']
delivered = orders.loc[orders['order_status']=='delivered',order_cols]


In [17]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [18]:
seller_orders = delivered.merge(items[['order_id','shipping_limit_date','seller_id']],on='order_id',how='left')

In [19]:
time_cols = ['order_purchase_timestamp','order_delivered_carrier_date','order_delivered_customer_date','shipping_limit_date']
for col in time_cols:
    seller_orders[col] = pd.to_datetime(seller_orders[col])


In [20]:
seller_orders = seller_orders.sort_values(by=['seller_id','order_purchase_timestamp'])

In [21]:
zero_time = timedelta(0)
seller_orders['is_shipped_on_time'] =  np.where((seller_orders['shipping_limit_date'] - seller_orders['order_delivered_carrier_date'])>zero_time,1,0)

In [22]:
# seller_timely_delivery_avg = (seller_orders.groupby('seller_id')['is_shipped_on_time']
#     .transform(lambda x: x.shift().rolling(window=25, min_periods=1).mean()))

In [23]:
seller_performance = (

    seller_orders.groupby('seller_id')['is_shipped_on_time']
    .apply(lambda x: x.tail(25).mean())
    .reset_index(name='seller_timely_delivery_avg')
)

In [24]:
seller_previous_order_count = seller_orders.groupby('seller_id')['order_id'].count()


In [25]:
seller_performance['seller_previous_order_count'] = seller_performance['seller_id'].map(seller_previous_order_count)

In [26]:
import os

path = os.path.join('..','data','processed')

shopping_dataset_unique.to_csv(os.path.join(path,'inp_form_orders_show.csv'),index=False)
unique_geo.to_csv(os.path.join(path,'unique_geo.csv'),index=False)
seller_performance.to_csv(os.path.join(path,'seller_performance.csv'),index=False)

In [27]:

unique_geo_df = unique_geo.reset_index()

In [28]:
unique_geo_df

,index,geolocation_zip_code_prefix,lat,lng,state
0,0,1001,-23.550107,-46.634027,SP
1,1,1002,-23.548228,-46.635247,SP
2,2,1003,-23.548976,-46.635318,SP
3,3,1004,-23.549550,-46.634771,SP
4,4,1005,-23.549690,-46.636532,SP
...,...,...,...,...,...
19010,19010,99960,-27.953797,-52.029641,RS
19011,19011,99965,-28.179542,-52.035551,RS
19012,19012,99970,-28.343257,-51.875470,RS
19013,19013,99980,-28.388342,-51.846871,RS


In [29]:
zip_code = 5221

In [30]:
if unique_geo[unique_geo['geolocation_zip_code_prefix'] == zip_code].shape[0]<1:
    print(f'{zip_code} not exist in database')

else:
    print(f'{zip_code} does exist in database')

5221 not exist in database


In [31]:
payments =  pd.read_csv('../data/raw/order_payments.csv')

In [32]:
payments['payment_type'].unique()

array(['credit_card', 'boleto', 'voucher', 'debit_card', 'not_defined'],
      dtype=object)